# 03 A/B 检验、区间与业务解释

主要指标 conversion；辅助指标 visit、含零值的人均 spend。详细假设在 docs/01-analysis-plan.md。

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / "src").exists():
    root = root.parent
sys.path.insert(0, str(root))
import pandas as pd
from src.data import download
from src.config import OUT
df = download()
print(f"客户数：{len(df):,}；字段数：{len(df.columns)}")

客户数：64,000；字段数：12


## 一、绝对与相对提升
概率差乘以 100 得到百分点；概率差除以对照率得到相对提升。这两个数不能混淆。

In [2]:
from src.statistics import rate_difference, ab_analysis, power_analysis
if not (OUT / 'ab_results.csv').exists():
    ab_analysis(df)
results = pd.read_csv(OUT / 'ab_results.csv')
print(results[['treatment','control','metric','effect','relative_lift','ci_low','ci_high','p_holm']].to_string(index=False))

    treatment       control     metric   effect  relative_lift   ci_low  ci_high        p_holm
  Mens E-Mail     No E-Mail conversion 0.006805       1.188422 0.005014 0.008641  1.218579e-12
  Mens E-Mail     No E-Mail      visit 0.076590       0.721405 0.069953 0.083227 6.822199e-111
  Mens E-Mail     No E-Mail      spend 0.769827       1.179289 0.485140 1.054515  6.890232e-07
Womens E-Mail     No E-Mail conversion 0.003111       0.543313 0.001502 0.004745  6.284205e-04
Womens E-Mail     No E-Mail      visit 0.045233       0.426055 0.038894 0.051574  3.182403e-43
Womens E-Mail     No E-Mail      spend 0.424412       0.650152 0.168957 0.679868  2.258794e-03
  Mens E-Mail Womens E-Mail conversion 0.003694       0.418003 0.001746 0.005663  6.284205e-04
  Mens E-Mail Womens E-Mail      visit 0.031356       0.207109 0.024284 0.038428  3.421949e-17
  Mens E-Mail Womens E-Mail      spend 0.345415       0.320660 0.032561 0.658268  3.046865e-02
   Any E-Mail     No E-Mail conversion 0.004955   

## 二、直接复算购买率差
二元结果使用两比例 z 检验和 Newcombe 区间。spend 使用 Welch，允许组间方差不同，并用 bootstrap 补充检查均值差区间。

In [3]:
treated = df.loc[df.segment == 'Mens E-Mail', 'conversion']
control = df.loc[df.segment == 'No E-Mail', 'conversion']
print(rate_difference(treated, control))
print('绝对百分点：', (treated.mean() - control.mean()) * 100)
print('相对提升：', (treated.mean() - control.mean()) / control.mean())

{'effect': 0.006805006519615695, 'ci_low': 0.005014344671481532, 'ci_high': 0.008640723396917826, 'se': 0.0009208727857351216, 'p_value': 1.5232235005955427e-13}
绝对百分点： 0.6805006519615695
相对提升： 1.188421876286328


## 三、多重比较与功效
12 项固定检验统一 Holm 修正；95% 普通区间为逐项覆盖。功效表是未来 MDE 场景规划，不是观测效应的事后功效。

In [4]:
power = power_analysis(df)
print(power.to_string(index=False))

 p0_planning_estimate  absolute_mde  ratio_t_to_c    alpha  power_target  n_control_required  n_treatment_required  scenario_power_at_current_n
             0.005726         0.002             1 0.050000           0.8               26074                 26074                     0.716535
             0.005726         0.005             1 0.050000           0.8                5003                  5003                     0.999934
             0.005726         0.010             1 0.050000           0.8                1571                  1571                     1.000000
             0.005726         0.002             1 0.004167           0.8               45647                 45647                     0.369670
             0.005726         0.005             1 0.004167           0.8                8758                  8758                     0.998231
             0.005726         0.010             1 0.004167           0.8                2750                  2750                     1